# NB09 — Musanga disturbance ecology
V5 · UAV-derived fractional canopy cover, not satellite predictions.

**Question:** What does present Musanga dominance reveal about disturbance legacies and spectral recovery? This is a cross-sectional chronosequence, not direct observation of species succession. No new imagery download or production model training. All analyses use the completed NB08bis Block 14 cache. HLS history begins in 2013; Hansen and JRC have different definitions and temporal coverage. No recorded event is not proof of undisturbed forest. Forest-specific results are primary; equal-forest summaries are conditional on these four forests, not representative of the Congo Basin.

In [ ]:
#@title 00 — Load frozen reference and historical datasets
from pathlib import Path
import json,hashlib,os
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if p.is_dir()),None)
assert ROOT is not None
RUN_REL='cfda7cd488492dd71a18/14_long_history/d669f0a998d16594f131/hansen_mask_corrected_v2/validation/a410ff2383b295dd2014' #@param {type:'string'}
R=ROOT/'Outputs/08bis_abundance_exploration_v5'/RUN_REL
assert (R/'completion.json').is_file(),'Complete NB08bis 14A–C first or update RUN_REL to the completed run'
manifest=json.loads((R/'completion.json').read_text())
assert manifest['complete'] and manifest['n']>0
F=R.parent/'features_50m.parquet'; A=R.parent.parent/'annual_HLS_50m.parquet'
REF=Path(manifest['reference'])
if not REF.exists(): REF=ROOT/'Outputs/08_satellite_reference'/REF.name
K=['site','mosaic','row','col']; Y='target_fraction'
SITES=['Baego','MOS','Yangambi','Yoko']; YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
BANDS=['NDVI','NDMI','NBR','NIR','SWIR1','SWIR2']
d=pd.read_parquet(REF,columns=K+['x','y',Y])
f=pd.read_parquet(F); a=pd.read_parquet(A)
assert not d.duplicated(K).any() and not f.duplicated(K).any() and len(d)==len(f)
D=d.merge(f,on=K,validate='one_to_one');assert len(D)==len(d) and D[Y].between(0,1).all()
assert not a.duplicated(K+['year']).any()
assert (a.year<a.site.map(YEARS)).all()
check=a.merge(d[K+[Y]],on=K,validate='many_to_one',suffixes=('','_ref'))
assert len(check)==len(a) and np.allclose(check[Y],check[Y+'_ref'])
del check
# Use source-specific cohorts; joint support is a sensitivity analysis, not the only ecology sample.
D['G_ok']=D.QA14_hansen_support.ge(.9)
D['J_ok']=D.QA14_jrc_support.ge(.9)
D['H_ok']=D.QA14_valid_years.ge(5)&D.QA14_recent_valid_years.ge(2)
D['joint']=D.eligible14.astype(bool)
assert D.joint.sum()==manifest['n']
D['block']=D.site.astype(str)+'_'+np.floor(D.x/500).astype(int).astype(str)+'_'+np.floor(D.y/500).astype(int).astype(str)
D['block1000']=D.site.astype(str)+'_'+np.floor(D.x/1000).astype(int).astype(str)+'_'+np.floor(D.y/1000).astype(int).astype(str)
D['cover_class']=pd.cut(D[Y],[-.001,.05,.2,.4,.6,1.001],labels=['0–5','5–20','20–40','40–60','60–100'],right=False)
D['cover_pp']=100*D[Y]
A=a.merge(D[K+['block','block1000','cover_class','G_ok','J_ok','H_ok','joint']],on=K,validate='many_to_one')
assert len(A)==len(a)
OUT=ROOT/'Outputs/09_disturbance_ecology_v5'/hashlib.sha256(RUN_REL.encode()).hexdigest()[:20];OUT.mkdir(parents=True,exist_ok=True)
(OUT/'input_contract.json').write_text(json.dumps(dict(reference=str(REF),features=str(F),annual=str(R.parent.parent/'annual_HLS_50m.parquet'),source_run=str(R),n=len(D),joint=int(D.joint.sum()),years=YEARS),indent=2))
print('Reference cells:',len(D),'Joint cohort:',D.joint.sum(),'Output:',OUT)
def save(df,name):
 df.to_csv(OUT/(name+'.csv'),index=False)
 return df
def figsave(fig,name):
 fig.savefig(OUT/(name+'.png'),dpi=220,bbox_inches='tight');fig.savefig(OUT/(name+'.pdf'),bbox_inches='tight');plt.show();plt.close(fig)


In [ ]:
#@title 01 — Sampling, support and spatial uncertainty
SEED=2271; BOOT=500; MIN_N=30; MIN_BLOCKS=5
def block_mean(g,value,block='block',seed=SEED):
 z=g[[value,block]].dropna();q=z.groupby(block)[value].agg(['sum','count']).to_numpy(float)
 n=len(z);b=len(q)
 if n<MIN_N or b<MIN_BLOCKS:return dict(n=n,blocks=b,mean=z[value].mean(),lo=np.nan,hi=np.nan,supported=False)
 rng=np.random.default_rng(seed)
 p=q[rng.integers(0,b,(BOOT,b))].sum(axis=1);v=p[:,0]/p[:,1]
 return dict(n=n,blocks=b,mean=z[value].mean(),lo=np.quantile(v,.025),hi=np.quantile(v,.975),supported=True)
def grouped(g,keys,value,block='block'):
 rows=[]
 for k,z in g.groupby(keys,observed=True,sort=True):
  if not isinstance(k,tuple):k=(k,)
  rows.append(dict(zip(keys,k))|block_mean(z,value,block))
 return pd.DataFrame(rows)
def equal_forest(g,keys):
 # Require all four forests; no changing composition hidden in pooled curves.
 rows=[]
 for k,z in g.groupby(keys,observed=True):
  if not isinstance(k,tuple):k=(k,)
  good=z[z.supported]
  rows.append(dict(zip(keys,k))|dict(forests=good.site.nunique(),mean=good['mean'].mean() if good.site.nunique()==4 else np.nan))
 return pd.DataFrame(rows)
qa=D.groupby('site').agg(n=(Y,'size'),blocks=('block','nunique'),cover_mean=('cover_pp','mean'),G=('G_ok','sum'),J=('J_ok','sum'),H=('H_ok','sum'),joint=('joint','sum')).reset_index()
display(save(qa,'sampling_support_by_forest'))
display(save(D.groupby(['site','cover_class'],observed=True).size().rename('n').reset_index(),'abundance_distribution'))
fig,axs=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for s,g in D.groupby('site'):
 axs[0].hist(g.cover_pp,bins=np.arange(0,105,5),density=True,histtype='step',label=s)
 axs[1].scatter(g.x,g.y,c=g.cover_pp,s=2,vmin=0,vmax=100,cmap='viridis',label=s)
axs[0].set(xlabel='UAV Musanga cover (%)',ylabel='Density');axs[0].legend();axs[1].set(title='Reference sampling footprints',aspect='equal');figsave(fig,'sampling')


In [ ]:
#@title 02 — Disturbance age, extent, recurrence and regrowth
# Hansen loss fraction is spatial extent at 50 m, not severity of canopy loss.
# Conditional age is loss-weighted mean; JRC duration is current regrowth duration.
LOSS_MIN=.2 #@param {type:'number'}
q=D[D.G_ok & D.G14_loss_any.ge(LOSS_MIN)].copy()
q['age']=pd.cut(q.G14_loss_age_conditional,[0,3,7,12,30],labels=['1–3','4–7','8–12','13+'])
D['loss_extent']=pd.cut(D.G14_loss_any,[-.001,0,.1,.25,.5,.75,1.001],labels=['none','0–10','10–25','25–50','50–75','75–100'])
D['jrc_entries']=pd.cut(D.J14_disturbance_entries,[-.001,0,.5,1,2,np.inf],labels=['0','0–0.5','0.5–1','1–2','>2'])
D['regrowth_age']=pd.cut(D.J14_regrowth_age_conditional,[0,3,7,12,20,40],labels=['1–3','4–7','8–12','13–20','21+'])
analyses=[('Hansen_age',q,'age'),('Hansen_extent',D[D.G_ok],'loss_extent'),('JRC_entries',D[D.J_ok],'jrc_entries'),('JRC_regrowth_age',D[D.J_ok&D.J14_last_regrowth.ge(.2)],'regrowth_age')]
for name,g,var in analyses:
 t=save(grouped(g,['site',var],'cover_pp'),name+'_by_forest')
 save(grouped(g,['site',var],'cover_pp','block1000'),name+'_1000m_sensitivity')
 save(equal_forest(t,[var]),name+'_equal_forest')
 fig,axs=plt.subplots(2,2,figsize=(11,7),layout='constrained',sharey=True)
 cats=list(g[var].cat.categories)
 for site,ax in zip(SITES,axs.ravel()):
  z=t[t.site.eq(site)]
  for _,r in z.iterrows():
   x=cats.index(r[var]);ax.plot(x,r['mean'],'o',color='tab:green' if r.supported else 'gray')
   if r.supported:ax.vlines(x,r.lo,r.hi,color='tab:green')
  ax.set(xticks=range(len(cats)),xticklabels=cats,title=site,ylabel='Mean Musanga cover (%)',ylim=(0,100));ax.tick_params(axis='x',rotation=20)
 figsave(fig,name)
# Joint age–extent response, supported bins only; no interpolation.
q['extent']=pd.cut(q.G14_loss_any,[0,.25,.5,.75,1.001],labels=['0–25','25–50','50–75','75–100'])
t=save(grouped(q,['site','age','extent'],'cover_pp'),'age_extent_by_forest')
fig,axs=plt.subplots(2,2,figsize=(10,8),layout='constrained')
for site,ax in zip(SITES,axs.ravel()):
 z=t[t.site.eq(site)&t.supported];p=z.pivot(index='extent',columns='age',values='mean').reindex(index=q.extent.cat.categories,columns=q.age.cat.categories)
 im=ax.imshow(p.to_numpy(float),vmin=0,vmax=100,origin='lower',cmap='YlGn');ax.set(xticks=range(4),xticklabels=p.columns,yticks=range(4),yticklabels=p.index,title=site,xlabel='Loss age (years)',ylabel='Loss extent (%)')
fig.colorbar(im,ax=axs,label='Mean cover (%)');figsave(fig,'age_extent')


In [ ]:
#@title 03 — HLS disturbance trajectories aligned to Hansen loss
# Use a dominant loss-year only if age-bin evidence and conditional mean agree tightly.
# Exact event alignment is reconstructed from corrected cached 30 m history TIFFs.
import rasterio
from rasterio.transform import rowcol
from collections import defaultdict
HIST=R.parent
# 50 m cell footprint exact overlap on the 10 m subgrid; Hansen pixels are 30 m.
def dominant_events():
 rows=[]
 for (site,tx,ty),g in D.assign(tx=np.floor(D.x/1500).astype(int),ty=np.floor(D.y/1500).astype(int)).groupby(['site','tx','ty']):
  p=HIST/f'{site}_{tx}_{ty}_history.tif'
  assert p.is_file(),p
  with rasterio.open(p) as ds:
   assert ds.shape==(50,50) and ds.descriptions[:2]==('treecover2000','lossyear')
   assert ds.crs.to_epsg()==32635
   tc,loss=ds.read([1,2]);tr=ds.transform
   assert np.isclose(tr.a,30) and np.isclose(-tr.e,30)
  valid=(tc>=0)&(tc<=100)&(loss>=0)&(loss<YEARS[site]-2000)
  v=np.repeat(np.repeat(valid,3,0),3,1);l=np.repeat(np.repeat(loss,3,0),3,1)
  for z in g.itertuples(index=False):
   rr=int(np.floor((tr.f-z.y)/50));cc=int(np.floor((z.x-tr.c)/50))
   assert 0<=rr<30 and 0<=cc<30
   a=l[rr*5:rr*5+5,cc*5:cc*5+5];ok=v[rr*5:rr*5+5,cc*5:cc*5+5]
   vals=a[ok&(a>0)].astype(int);n=ok.sum()
   if n<23 or not len(vals):continue
   years,counts=np.unique(vals,return_counts=True);k=np.argmax(counts)
   rows.append(dict(site=site,mosaic=z.mosaic,row=z.row,col=z.col,event_year=2000+int(years[k]),event_share=counts[k]/n,loss_share=len(vals)/n,dominance=counts[k]/len(vals)))
 return pd.DataFrame(rows)
E=dominant_events();save(E,'dominant_hansen_events')
E=E[E.event_share.ge(.2)&E.dominance.ge(.7)]
T=A.merge(E,on=K,validate='many_to_one')
T['relative_year']=T.year-T.event_year
# Fixed pre/post availability window to avoid changing cohort composition.
wide=T.pivot(index=K,columns='relative_year',values='NBR')
needed=[-2,-1,0,1,2]
if set(needed)<=set(wide.columns):
 cohort=wide.dropna(subset=needed).reset_index()[K]
else:cohort=wide.iloc[:0].reset_index()[K]
T=T.merge(cohort,on=K,validate='many_to_one')
base=T[T.relative_year.isin([-2,-1])].groupby(K)[BANDS].mean().add_suffix('_pre').reset_index()
T=T.merge(base,on=K,validate='many_to_one')
for b in BANDS:T[b+'_delta']=T[b]-T[b+'_pre']
save(T.groupby(['site','cover_class'],observed=True)[Y].count().rename('annual_rows').reset_index(),'event_aligned_support')
for b in ['NBR','NDMI','NDVI']:
 z=grouped(T[T.relative_year.between(-2,2)],['site','cover_class','relative_year'],b+'_delta')
 save(z,'event_aligned_'+b)
 fig,axs=plt.subplots(2,2,figsize=(11,7),layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  for c,g in z[z.site.eq(s)&z.supported].groupby('cover_class',observed=True):
   g=g.sort_values('relative_year');ax.plot(g.relative_year,g['mean'],'o-',label=str(c));ax.fill_between(g.relative_year,g.lo,g.hi,alpha=.15)
  ax.axvline(0,color='gray',ls='--');ax.axhline(0,color='gray',lw=.5);ax.set(title=s,xlabel='Years relative to Hansen loss',ylabel=b+' change from pre-loss');ax.legend(fontsize=8)
 figsave(fig,'event_aligned_'+b)
print('Dominant event cells with complete −2…+2 observations:',len(cohort))


In [ ]:
#@title 04 — HLS history versus observed Musanga abundance
# Survey-relative trajectories on fixed recent 3-year support, and full available history separately.
for b in BANDS:
 g=A[A[b].notna()].copy()
 for cohort_name,sub in [('available',g),('joint',g[g.joint])]:
  t=grouped(sub,['site','cover_class','lag'],b);save(t,f'survey_{b}_{cohort_name}')
  if cohort_name!='available':continue
  fig,axs=plt.subplots(2,2,figsize=(11,7),layout='constrained')
  for s,ax in zip(SITES,axs.ravel()):
   for c,z in t[t.site.eq(s)&t.supported].groupby('cover_class',observed=True):
    z=z.sort_values('lag',ascending=False);ax.plot(-z.lag,z['mean'],'o-',label=str(c));ax.fill_between(-z.lag,z.lo,z.hi,alpha=.15)
   ax.set(title=s,xlabel='Years before UAV survey',ylabel=b);ax.legend(fontsize=8)
  figsave(fig,'survey_'+b)
# Forest-specific rank associations: descriptive, no naive cell-level p-values.
from scipy.stats import spearmanr
rows=[]
for site,g in D.groupby('site'):
 for c in [x for x in D if x.startswith(('G14_','J14_','H14_'))]:
  ok=g.G_ok if c.startswith('G14_') else g.J_ok if c.startswith('J14_') else g.H_ok
  z=g.loc[ok,[c,Y,'block']].dropna()
  if len(z)<MIN_N or z.block.nunique()<MIN_BLOCKS or z[c].nunique()<2:continue
  # Equal-sized block means provide a second, coarser-scale diagnostic.
  b=z.groupby('block')[[c,Y]].mean()
  rows.append(dict(site=site,feature=c,n=len(z),blocks=len(b),rho=spearmanr(z[c],z[Y]).statistic,block_rho=spearmanr(b[c],b[Y]).statistic))
display(save(pd.DataFrame(rows),'rank_associations').sort_values(['site','rho'],ascending=[True,False]).groupby('site').head(8))


In [ ]:
#@title 05 — Conditional ecological response (fractional-logit spline models)
# Interpretable conditional associations, not causal effects. Fit only on detected-loss cohort.
# Fixed forest effects; equal total forest weights; cluster bootstrap for response curves.
import statsmodels.api as sm
import patsy
z=D[D.joint&D.G14_loss_any.ge(.2)].copy()
z=z.rename(columns={'G14_loss_age_conditional':'age','G14_loss_any':'extent','J14_disturbance_entries':'recurrence','H14_NBR_recovery_slope':'recovery','H14_NBR_recent3':'recent_NBR'})
cols=['age','extent','recurrence','recovery','recent_NBR',Y,'site','block']
z=z[cols].dropna().reset_index(drop=True)
assert len(z)>100 and z.site.nunique()>=2,'Insufficient complete case support for conditional model'
print('Complete-case conditional model:',z.groupby('site').size().to_dict())
formulas={'disturbance':'bs(age,df=4)+bs(extent,df=4)+bs(recurrence,df=3)+C(site)','plus_HLS':'bs(age,df=4)+bs(extent,df=4)+bs(recurrence,df=3)+bs(recovery,df=4)+bs(recent_NBR,df=4)+C(site)'}
models={}
for name,formula in formulas.items():
 X=patsy.dmatrix(formula,z,return_type='dataframe')
 w=1/z.groupby('site').site.transform('size');w=w/w.mean()
 m=sm.GLM(z[Y],X,family=sm.families.Binomial(),freq_weights=w).fit(maxiter=100)
 models[name]=(m,X.design_info)
 print(name,'weighted deviance',m.deviance,'converged',m.converged)
 save(pd.DataFrame({'term':X.columns,'coefficient':m.params.to_numpy()}),'conditional_'+name+'_coefficients')
# Forest-specific empirical support; curves standardized to observed covariates within each forest.
rows=[]
for name,(m,info) in models.items():
 for site,g in z.groupby('site'):
  for age in np.linspace(g.age.quantile(.05),g.age.quantile(.95),25):
   local=g[g.age.sub(age).abs()<=2]
   if len(local)<MIN_N or local.block.nunique()<MIN_BLOCKS:continue
   # Standardize within local-age support to limit implausible extrapolation.
   Xnew=patsy.build_design_matrices([info],local.assign(age=age))[0]
   rows.append(dict(model=name,site=site,age=age,mean=100*np.mean(m.predict(Xnew)),n=len(local),blocks=local.block.nunique()))
p=save(pd.DataFrame(rows),'conditional_age_curves')
fig,axs=plt.subplots(2,2,figsize=(11,7),layout='constrained',sharey=True)
for site,ax in zip(SITES,axs.ravel()):
 for name,g in p[p.site.eq(site)].groupby('model'):
  ax.plot(g.age,g['mean'],label=name)
 ax.set(title=site,xlabel='Conditional loss age (years)',ylabel='Estimated Musanga cover (%)',ylim=(0,100));ax.legend()
figsave(fig,'conditional_age_curves')
# Model curves are exploratory point estimates, not confidence bands; use block-bootstrap descriptive figures for uncertainty.


## Interpretation checklist
- Separate Hansen loss extent from HLS spectral-drop magnitude and JRC disturbance-state entries (not necessarily independent events).
- Inspect forest-specific support and 500 m versus 1 km spatial-block sensitivity before interpreting differences.
- No-event cells may contain old disturbance, subpixel degradation or undetected events.
- Dominant Hansen event alignment uses only sufficiently homogeneous loss-year cells with complete pre/post annual NBR support; event year is approximate.
- HLS spectral recovery is not evidence of floristic or biomass recovery. UAV cover is a single-date estimated reference; sampling and UAV errors remain.
- The spline models are conditional descriptive associations; no causal mediation, no independent four-forest population inference. Future work: trajectory clustering, matched age/extent contrasts, conditional model block-bootstrap, explicit event type and patch context.